# 01 — Rol 1: Teoría de la información

**Daniel Figueredo (332679)**

Cuantificamos qué tan predecible es la demanda de UrbanMove y cuánto la reduce conocer el
momento (hora, día laboral/fin de semana). Los 8 conceptos exigidos por la guía se aplican
encadenados: cada uno responde una pregunta que deja abierta el anterior.

Los agregados se calculan sobre la **población limpia** (1,438,943 viajes), no sobre la
muestra: una prueba de permutación (sección 4) mostró que en la muestra el 54% de la
información mutua estimada era sesgo estadístico, no señal real.

In [1]:
import pandas as pd, numpy as np
LN2 = np.log(2)

P = pd.read_parquet('train_limpio.parquet')
P['hour'] = P.pickup_datetime.dt.hour
P['dow']  = P.pickup_datetime.dt.dayofweek
P['is_weekend'] = (P.dow >= 5).astype(int)

# Zona con el mismo grid de 1 km que usan los Roles 2 y 3 (src/config.py)
import importlib.util
spec = importlib.util.spec_from_file_location("config", "config.py")
cfgmod = importlib.util.module_from_spec(spec); spec.loader.exec_module(cfgmod)
R, LAT_C, LON_C, LADO = cfgmod.R_TIERRA_KM, cfgmod.LAT_C, cfgmod.LON_C, cfgmod.LADO_ZONA_KM

cl = np.radians(LAT_C)
def proyectar(lon, lat):
    return np.radians(lon-LON_C)*R*np.cos(cl), np.radians(lat-LAT_C)*R
P['px'], P['py'] = proyectar(P.pickup_longitude, P.pickup_latitude)
P['dx'], P['dy'] = proyectar(P.dropoff_longitude, P.dropoff_latitude)
zona = lambda x, y: np.floor(x/LADO).astype(int).astype(str) + '_' + np.floor(y/LADO).astype(int).astype(str)
P['zona_pick'] = zona(P.px, P.py)
P['zona_drop'] = zona(P.dx, P.dy)
print(f"Zonas de recogida: {P.zona_pick.nunique()} | zonas de destino: {P.zona_drop.nunique()}")


Zonas de recogida: 603 | zonas de destino: 1123


### 1-2. Probabilidad (MLE) y Entropía de Shannon

In [2]:
def H(counts):
    c = np.asarray(counts, float); c = c[c > 0]; p = c/c.sum()
    return float(-(p*np.log2(p)).sum())

pz = P.zona_pick.value_counts(normalize=True)
print(f"Top 10 zonas concentran {pz.head(10).sum()*100:.2f}% de las recogidas (probabilidad MLE)")

h_pick, h_drop = H(P.zona_pick.value_counts()), H(P.zona_drop.value_counts())
print(f"H(zona recogida) = {h_pick:.3f} bits  -> {2**h_pick:.1f} zonas efectivas de {P.zona_pick.nunique()}")
print(f"H(zona destino)  = {h_drop:.3f} bits  -> {2**h_drop:.1f} zonas efectivas de {P.zona_drop.nunique()}")


Top 10 zonas concentran 46.91% de las recogidas (probabilidad MLE)
H(zona recogida) = 5.466 bits  -> 44.2 zonas efectivas de 603
H(zona destino)  = 6.035 bits  -> 65.6 zonas efectivas de 1123


### 3-4. Entropía condicional e Información mutua

In [3]:
def H_cond(x, y):
    t = pd.crosstab(y, x).values
    return float(sum(t[i].sum()/t.sum()*H(t[i]) for i in range(t.shape[0])))
def MI(x, y): return H(pd.Series(x).value_counts()) - H_cond(x, y)

for nm, col in [('recogida', 'zona_pick'), ('destino', 'zona_drop')]:
    hc = H_cond(P[col], P.hour)
    mi = MI(P[col], P.hour)
    h0 = h_pick if col == 'zona_pick' else h_drop
    print(f"H(zona {nm}|hora) = {hc:.3f} bits | I(hora;zona {nm}) = {mi:.4f} bits ({mi/h0*100:.2f}% de H)")


H(zona recogida|hora) = 5.385 bits | I(hora;zona recogida) = 0.0813 bits (1.49% de H)


H(zona destino|hora) = 5.920 bits | I(hora;zona destino) = 0.1144 bits (1.90% de H)


In [4]:
# La información mutua estimada por conteo tiene sesgo positivo. Lo medimos con una
# prueba de permutación sobre la MUESTRA (donde el efecto es más visible por el tamaño n).
muestra = pd.read_parquet('muestra_50k.parquet')
muestra['px'], muestra['py'] = proyectar(muestra.pickup_longitude, muestra.pickup_latitude)
muestra['dx'], muestra['dy'] = proyectar(muestra.dropoff_longitude, muestra.dropoff_latitude)
muestra['zona_drop'] = zona(muestra.dx, muestra.dy)
muestra['hour'] = muestra.pickup_datetime.dt.hour

rng = np.random.default_rng(42)
obs = MI(muestra.zona_drop, muestra.hour)
null = np.array([MI(muestra.zona_drop, rng.permutation(muestra.hour.values)) for _ in range(200)])
print(f"I(hora;zona destino) en la MUESTRA: observado {obs:.4f} | promedio bajo azar (200 permutaciones) {null.mean():.4f}")
print(f"Sesgo: {null.mean()/obs*100:.1f}% del valor observado. Por eso las cifras oficiales usan la población.")


I(hora;zona destino) en la MUESTRA: observado 0.2167 | promedio bajo azar (200 permutaciones) 0.1177
Sesgo: 54.3% del valor observado. Por eso las cifras oficiales usan la población.


### 5-6. Divergencia KL y Entropía cruzada — laboral vs. fin de semana

In [5]:
def KL(pc, qc, alpha=0.5):
    idx = pc.index.union(qc.index)
    p = pc.reindex(idx, fill_value=0)+alpha; q = qc.reindex(idx, fill_value=0)+alpha
    p, q = p/p.sum(), q/q.sum()
    return float((p*np.log2(p/q)).sum())

L, W = P[P.is_weekend==0], P[P.is_weekend==1]
kl_hora = KL(W.hour.value_counts(), L.hour.value_counts(), alpha=0)
kl_zona = KL(W.zona_pick.value_counts(), L.zona_pick.value_counts())
print(f"KL(finde||laboral) en HORA: {kl_hora:.4f} bits")
print(f"KL(finde||laboral) en ZONA de recogida: {kl_zona:.4f} bits")
print(f"-> el fin de semana difiere sobre todo en horario, casi nada en el mapa")

# Entropía cruzada = entropía + KL: costo de planificar el finde con el modelo laboral
h_finde_hora = H(W.hour.value_counts())
print(f"Costo de aplicar el modelo laboral al horario de fin de semana: +{kl_hora/h_finde_hora*100:.2f}%")


KL(finde||laboral) en HORA: 0.1815 bits
KL(finde||laboral) en ZONA de recogida: 0.0549 bits
-> el fin de semana difiere sobre todo en horario, casi nada en el mapa
Costo de aplicar el modelo laboral al horario de fin de semana: +4.06%


### 7. Desigualdad de Jensen — dos aplicaciones

In [6]:
# (a) Concavidad de la entropía: explica por qué I nunca es negativa
print(f"H(zona) = {h_pick:.4f}  >=  H(zona|hora) = {H_cond(P.zona_pick, P.hour):.4f}")
print(f"Brecha = {h_pick - H_cond(P.zona_pick, P.hour):.4f} bits = I(hora;zona) exacto")

# (b) El logaritmo es cóncavo: la media geométrica (típica) subestima el promedio real
dur_min = muestra.trip_duration / 60
tipico = np.exp(np.log(dur_min).mean())
promedio = dur_min.mean()
print(f"\nDuración: típica (media geométrica) = {tipico:.2f} min | promedio real = {promedio:.2f} min")
print(f"Brecha: {(promedio/tipico - 1)*100:.1f}% — el Rol 3 debe reportar 'típico', no 'promedio'")


H(zona) = 5.4664  >=  H(zona|hora) = 5.3851
Brecha = 0.0813 bits = I(hora;zona) exacto

Duración: típica (media geométrica) = 10.84 min | promedio real = 13.99 min
Brecha: 29.1% — el Rol 3 debe reportar 'típico', no 'promedio'


### 8. Log-sum-exp — clasificar el tipo de día (laboral / fin de semana) sin desbordar

In [7]:
from scipy.special import logsumexp

muestra['celda'] = muestra.zona_pick.astype(str) if 'zona_pick' in muestra else zona(muestra.px, muestra.py)
muestra['zona_pick'] = zona(muestra.px, muestra.py)
muestra['celda'] = muestra.zona_pick + '|' + muestra.hour.astype(str)
muestra['dia'] = muestra.pickup_datetime.dt.date
cod, celdas = pd.factorize(muestra.celda); muestra['c'] = cod; K = len(celdas)
ALPHA = 0.5
is_weekend_m = (muestra.pickup_datetime.dt.dayofweek >= 5).astype(int)
cnt = {cl: np.bincount(muestra.c[is_weekend_m==cl], minlength=K).astype(float) for cl in (0, 1)}
dias = muestra.groupby('dia').agg(n=('c', 'size'))
dias['finde'] = muestra.groupby('dia').apply(lambda d: is_weekend_m.loc[d.index].iloc[0])
prior = np.log(np.array([(dias.finde==0).mean(), (dias.finde==1).mean()]))

aciertos = 0
for d, r in dias.iterrows():
    c_dia = muestra.c[muestra.dia==d].values
    ll = []
    for cl in (0, 1):
        base = cnt[cl] - (np.bincount(c_dia, minlength=K) if cl == r.finde else 0)
        logp = np.log((base+ALPHA)/(base.sum()+ALPHA*K))
        ll.append(logp[c_dia].sum())
    ll = np.array(ll) + prior
    pred = int(np.argmax(ll))
    aciertos += (pred == r.finde)

print(f"Acierto clasificando {len(dias)} días (dejando cada uno fuera de su propio modelo): "
      f"{aciertos}/{len(dias)} = {aciertos/len(dias)*100:.2f}%")
print(f"Referencia (predecir siempre 'laboral'): {(dias.finde==0).mean()*100:.2f}%")
print("\nEl producto directo de probabilidades desborda a 0.0 con ~270 viajes/día;")
print("log-sum-exp evita el problema calculando siempre en escala logarítmica.")


Acierto clasificando 182 días (dejando cada uno fuera de su propio modelo): 178/182 = 97.80%
Referencia (predecir siempre 'laboral'): 71.43%

El producto directo de probabilidades desborda a 0.0 con ~270 viajes/día;
log-sum-exp evita el problema calculando siempre en escala logarítmica.


## Resumen para la integración (Rol 4)

- La hora reduce poco la incertidumbre espacial: I(hora;zona destino) ≈ 0.11 bits (≈1.9% de H).
- El fin de semana difiere sobre todo en **horario** (KL=0.18 bits), casi nada en **mapa** (KL=0.05).
- El clasificador de log-sum-exp identifica festivos (Año Nuevo, Memorial Day) sin que se le digan.
- Puente con el Rol 3: reportar duraciones **típicas** (media geométrica), no promedio.